In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

GOLDEN_DIR = Path("../data/golden")
REPORT_DIR = Path("../reports")

accounts = pd.read_csv(GOLDEN_DIR / "accounts_golden.csv")
payments = pd.read_csv(GOLDEN_DIR / "payments_golden.csv")
calls = pd.read_csv(GOLDEN_DIR / "calls_golden.csv")
attempts = pd.read_csv(GOLDEN_DIR / "call_attempts_golden.csv")
targeting = pd.read_csv(GOLDEN_DIR / "daily_targeting_golden.csv")

In [2]:
recovery = (
    payments[payments["payment_status"] == "SUCCESS"]
    .groupby("account_id")["amount"]
    .sum()
    .reset_index(name="recovery")
)

account_analysis = accounts.merge(
    recovery,
    on="account_id",
    how="left"
)

account_analysis["recovery"] = (
    account_analysis["recovery"].fillna(0)
)

account_analysis["recovered"] = (
    account_analysis["recovery"] > 0
)

In [3]:
dpd = (
    account_analysis
    .groupby("dpd")
    .agg(
        accounts=("account_id", "nunique"),
        recovered=("recovered", "sum"),
        recovery=("recovery", "sum")
    )
    .reset_index()
)

dpd["recovery_rate"] = (
    dpd["recovered"] / dpd["accounts"] * 100
)

dpd["recovery_per_account"] = (
    dpd["recovery"] / dpd["accounts"]
)

display(dpd)

,dpd,accounts,recovered,recovery,recovery_rate,recovery_per_account
0,0,2685,1220,1.175878e+08,45.437616,43794.356842
1,1,2713,1161,1.171067e+08,42.793955,43165.006141
2,5,2727,1222,1.232974e+08,44.811148,45213.572871
3,15,2736,1178,1.163179e+08,43.055556,42513.844967
4,30,2704,1158,1.149939e+08,42.825444,42527.340203
5,45,2744,1257,1.231857e+08,45.809038,44892.748138
6,60,2770,1267,1.298257e+08,45.740072,46868.474809
7,75,2741,1215,1.199576e+08,44.326888,43764.172273
8,90,2727,1221,1.188740e+08,44.774477,43591.501236
9,120,2759,1225,1.236570e+08,44.400145,44819.506720


In [4]:
for col in ["client", "geography", "language", "risk_segment"]:
    if col in account_analysis.columns:
        result = (
            account_analysis
            .groupby(col, dropna=False)
            .agg(
                accounts=("account_id", "nunique"),
                recovered=("recovered", "sum"),
                recovery=("recovery", "sum")
            )
            .reset_index()
        )

        result["recovery_rate"] = (
            result["recovered"]
            / result["accounts"]
            * 100
        )

        print(f"\n===== {col.upper()} =====")
        display(result.sort_values("recovery", ascending=False))


===== RISK_SEGMENT =====


,risk_segment,accounts,recovered,recovery,recovery_rate
0,HIGH,7552,3300,3.317907e+08,43.697034
1,LOW,7513,3381,3.311408e+08,45.001997
2,MEDIUM,7533,3335,3.309252e+08,44.271870
3,NPA,7402,3268,3.226178e+08,44.150230


In [5]:
campaign = calls.merge(
    targeting[
        ["account_id", "campaign_id"]
    ].drop_duplicates(),
    on="account_id",
    how="left",
    suffixes=("_call", "_target")
)

campaign["campaign"] = (
    campaign["campaign_id_call"]
    .fillna(campaign["campaign_id_target"])
)

campaign_recovery = campaign.merge(
    recovery,
    on="account_id",
    how="left"
)

campaign_recovery["recovery"] = (
    campaign_recovery["recovery"].fillna(0)
)

display(
    campaign_recovery
    .groupby("campaign")
    .agg(
        accounts=("account_id", "nunique"),
        recovery=("recovery", "sum"),
        calls=("call_id", "nunique")
    )
    .sort_values("recovery", ascending=False)
    .head(20)
)

,accounts,recovery,calls
campaign,,,
CMP0000036,781,66359367.77,791
CMP0000006,740,65583956.65,749
CMP0000072,777,65573915.22,789
CMP0000116,804,65170485.98,815
CMP0000055,750,64606252.49,756
CMP0000048,775,63623622.32,788
CMP0000098,768,63337199.83,773
CMP0000040,765,62829547.99,771
CMP0000085,731,62759451.79,738


In [6]:
vendor = (
    calls
    .groupby("vendor_id")
    .agg(
        calls=("call_id", "nunique"),
        answered=("call_status",
                   lambda x: (x == "ANSWERED").sum()),
        avg_duration=("duration_sec", "mean")
    )
    .reset_index()
)

vendor["answer_rate"] = (
    vendor["answered"] / vendor["calls"] * 100
)

display(
    vendor.sort_values(
        "answer_rate",
        ascending=False
    )
)

,vendor_id,calls,answered,avg_duration,answer_rate
13,VND0000014,6041,1275,452.632484,21.105777
5,VND0000006,5896,1230,450.894460,20.861601
9,VND0000010,6163,1251,453.707962,20.298556
10,VND0000011,6072,1222,450.903422,20.125165
12,VND0000013,5904,1188,453.300389,20.121951
2,VND0000003,5964,1188,454.254442,19.919517
7,VND0000008,5911,1168,448.908062,19.759770
1,VND0000002,5934,1170,443.459519,19.716886
14,VND0000015,6066,1195,454.626915,19.699967
3,VND0000004,5925,1157,450.662732,19.527426


In [7]:
calls["event_at"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

calls["hour"] = calls["event_at"].dt.hour

time_analysis = (
    calls
    .groupby("hour")
    .agg(
        calls=("call_id", "nunique"),
        answered=("call_status",
                   lambda x: (x == "ANSWERED").sum())
    )
    .reset_index()
)

time_analysis["answer_rate"] = (
    time_analysis["answered"]
    / time_analysis["calls"]
    * 100
)

display(time_analysis)

,hour,calls,answered,answer_rate
0,0,3629,738,20.336181
1,1,3745,768,20.507343
2,2,3878,723,18.643631
3,3,3775,755,20.000000
4,4,3697,747,20.205572
5,5,3797,729,19.199368
6,6,3815,755,19.790301
7,7,3793,748,19.720538
8,8,3700,699,18.891892
9,9,3699,767,20.735334


In [8]:
attempts_per_account = (
    attempts
    .groupby("account_id")
    .size()
    .reset_index(name="attempts")
)

attempts_per_account["attempt_bucket"] = pd.cut(
    attempts_per_account["attempts"],
    bins=[0, 1, 3, 5, 10, np.inf],
    labels=["1", "2-3", "4-5", "6-10", "11+"]
)

attempt_analysis = account_analysis.merge(
    attempts_per_account,
    on="account_id",
    how="left"
)

attempt_analysis["attempts"] = (
    attempt_analysis["attempts"].fillna(0)
)

attempt_analysis["attempt_bucket"] = (
    attempt_analysis["attempt_bucket"]
    .cat.add_categories("0")
    .fillna("0")
)

result = (
    attempt_analysis
    .groupby("attempt_bucket")
    .agg(
        accounts=("account_id", "nunique"),
        recovered=("recovered", "sum"),
        recovery=("recovery", "sum")
    )
    .reset_index()
)

result["recovery_rate"] = (
    result["recovered"]
    / result["accounts"]
    * 100
)

result["recovery_per_account"] = (
    result["recovery"]
    / result["accounts"]
)

display(result)

C:\Users\ridhi\AppData\Local\Temp\ipykernel_33764\298431435.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("attempt_bucket")


,attempt_bucket,accounts,recovered,recovery,recovery_rate,recovery_per_account
0,1,2124,902,9.039270e+07,42.467043,42557.770490
1,2-3,10377,4615,4.577129e+08,44.473355,44108.399756
2,4-5,10553,4677,4.647381e+08,44.319151,44038.485659
3,6-10,6310,2818,2.766565e+08,44.659271,43844.137751
4,11+,87,32,3.483328e+06,36.781609,40038.258391
5,0,549,240,2.349092e+07,43.715847,42788.558288


In [9]:
dpd.to_csv(
    REPORT_DIR / "phase5_dpd.csv",
    index=False
)

vendor.to_csv(
    REPORT_DIR / "phase5_vendor.csv",
    index=False
)

time_analysis.to_csv(
    REPORT_DIR / "phase5_calling_time.csv",
    index=False
)

result.to_csv(
    REPORT_DIR / "phase5_attempt_frequency.csv",
    index=False
)

print("Phase 5 complete.")

Phase 5 complete.
